# TDM Capteurs: Représentation de l'environnement 3D

---

Aujourd'hui, pas d'acquisition, vous êtes déjà experts dans le domaine. Place à la représentation de l'environnement 3D !

*"Cool ! Mais ça veut dire quoi ?!"*

Derrière cet intitulé se cache la visualisation de l'environnement à partir des données acquises. Concrètement, on va exploiter les acquisitions de différents capteurs, et les transformer en une seule représentation riche en information. Bon, si vous ne voyez toujours pas, on va voir ça ensemble durant ces 3h ;)

In [ ]:
# Chargement des librairies
import cv2
import numpy as np
from matplotlib import pyplot as plt
import plotly.graph_objects as go

---

## 1) Parcours des données

Ca va vous étonner, mais pour exploiter des données, il nous faut des données. Et pour cela, on va utiliser des données du dataset [KITTI](https://www.cvlibs.net/datasets/kitti/index.php). 

Le dataset KITTI est un dataset de référence dans le domaine de la mobilité intelligente. De nombreuses acquisitions ont été réalisées, et traitées pour offrir des données de très bonne qualité provenant de plusieurs capteurs. Avec ces données, des "labels" (des annotations humaines) sont fournies pour plusieurs challenges, comme par exemple:
- Les objets présents dans l'acquisition et leur position (voiture, piéton, cycliste, etc.)
- La segmentation de la voie navigable (la zone pixel par pixel de la route)
- La carte de disparité d'une scène (la profondeur des objets pixel par pixel)

Bref, ce dataset est une mine d'or !

*Croyez-en mon expérience, des datasets aussi propres et complets, ça ne court pas les rues...*

**_QUESTION :_** Quel est le système de capteurs utilisé pour les acquisitions ?

**_REPONSE :_**

**_QUESTION :_** Comment sont placés les capteurs au niveau du véhicule ? Dans quelles orientations ?

**_REPONSE :_**

Pour accéder aux données, il vous faut un compte (en réalité, c'est pas compliqué, mais ça demande l'approbation de l'équipe de recherche à l'origine du dataset). Téléchargez et décompressez plutôt les archives suivantes disponibles sur Moodle :
- *2011_09_26_drive_0005_extract.zip*
- *2011_09_26_drive_0005_sync.zip*
- *2011_09_26_drive_0005_calib.zip*

On va commencer avec la première archive (*extract*).

**_QUESTION :_** Quelles sont les données disponibles ?

**_REPONSE :_**

**_QUESTION :_** A partir des informations disponibles, estimez les fréquences d'acquisition des différents capteurs

**_REPONSE :_**

**_QUESTION :_** Concernant les données GPS et IMU, que remarquez vous ?

**_REPONSE :_**

Les données que vous avez ici sont brutes (à un détail prêt), un peu comme celles que vous avec votre système d'acquisition. Mais des données brutes sans post-traitement, c'est bien, mais ça ne sert à rien. Heuresuement, l'équipe de recherche à la conception de KITTI nous fournit également les données traitées, comprenant:
- La synchronisation des données pour tous les capteurs
- La rectification des images (utile pour faire de la stéréovision)
- Le post-traitement des nuages de points LiDAR (compensation temporelle, élimination de points aberrants)

Toutes ces données traitées sont dans l'archive *sync*. 

**_QUESTION :_** Combien reste-t-il d'échantillons après synchronisation ?

**_REPONSE :_**

---

## 2) Calibration des données

Concentrons nous maintenant sur la dernière archive : *calib*. Vous y trouverez 3 fichiers txt comprenant plusieurs informations de calibration.

**_QUESTION :_** Que contient le fichier *calib_imu_to_velo.txt* ? A quoi servent les variables décrites ?

**_REPONSE :_**

**_QUESTION :_** Que contient le fichier *calib_velo_to_cam.txt* ? A quoi servent les variables décrites ?

*Note : les variables delta_f et delta_c sont des corrections sur la focale et le centre de la caméra, mais on ne va pas les utiliser*

**_REPONSE :_**

Passons sur le dernier fichier, mais pas des moindres: *calib_cam_to_cam.txt*. Vous avez ici plein de variables, correspondant aux 4 caméras (d'identifiants *xx*):
- *00* : Caméra gauche noir et blanc
- *01* : Caméra droite noir et blanc
- *02* : Caméra gauche couleur
- *03* : Caméra droite couleur

**_QUESTION :_** A quoi correspondent les matrices suivantes ?
- *S_xx*
- *K_xx*
- *D_xx*
- *R_xx*
- *T_xx*

**_REPONSE :_**

**_QUESTION :_** Au vu des différentes matrices, quelle caméra fait office de caméra de référence ?

**_REPONSE :_**

Vous avez pu constater également la présence de 3 variables supplémentaires pour chaque caméra (*S_rect_xx*, *R_rect_xx*, et *P_rect_xx*). Ces variables correspondent aux paramètres de calibration après rectification des images. On les laisse de côté pour l'instant, mais on les utilisera plus tard.

Pour résumer, toutes ces matrices permettent de passer de n'importe quel repère d'un capteur à un autre.

![](schema_capteurs.png)

On va pouvoir charger toutes ces matrices de calibration. Pour cela, utilisez la classe *Calibration*, du fichier *utils.py*

In [ ]:
from utils import Calibration

# A COMPLETER
# Chargement des données de calibration
...

Pour être sûr que vous avez bien compris comment est construit cette classe, affichez successivement:
- *T_velo_to_cam* (la matrice de translation LiDAR -> Caméra)
- *R_imu_to_velo* (la matrice de rotation IMU -> LiDAR)
- *R_02*
- *T_03*
- *K_01*

In [ ]:
# A COMPLETER
# Affichage des matrices de calibration demandées
...

---

## 3) Changement de repère

Ok, vous avez vos matrices de calibration à disposition, mais il faut savoir les utiliser ! Pour cela, on va faire quelques petits exercices de changement de repère.

Construisez dans un premier temps un point $M_{imu}$ placé dans le repère IMU:

\begin{equation*}
    M_{imu} = \begin{pmatrix} x_{imu} \\ y_{imu} \\ z_{imu} \end{pmatrix}
\end{equation*}

Avec $x_{imu} = 0$, $y_{imu} = 0$ et $z_{imu} = 0$

In [ ]:
# A COMPLETER
# Construction du point M_IMU
...

Pour exprimer ce point $M_{imu}$ dans le repère LiDAR afin d'obtenir $M_{velo}$, il va nous falloir utiliser les matrices de rotation et de translation pour passer du repère IMU au repère LiDAR:

\begin{equation*}
    \begin{pmatrix} x_{velo} \\ y_{velo} \\ z_{velo} \\ 1 \end{pmatrix} = \begin{pmatrix} R_{imu\_to\_velo} & T_{imu\_to\_velo} \\ 0_3 & 1 \end{pmatrix} \cdot \begin{pmatrix} x_{imu} \\ y_{imu} \\ z_{imu} \\ 1 \end{pmatrix}
\end{equation*}

Vous noterez ici que les points $M_{imu}$ et $M_{velo}$ sont exprimés en **coordonnées homogènes** avec $w = 1$.

Calculez le point $M_{velo}$

In [ ]:
# A COMPLETER
# Calcul de M_VELO
...

En suivant le même principe, calculez le point $M_{cam\_03}$, le même point $M$ exprimé dans le repère de la caméra droite couleur.

*Note : le changement de repère ne se fait pas directement, il faudra forcément passer par un autre repère...*

In [ ]:
# A COMPLETER
# Calcul de M_CAM_03
...

La prochaine étape serait ici de repérer ce point dans le plan image de la caméra... Cependant, vous devriez avoir ici $z_{cam\_03}$ négative. C'est logique, car l'IMU se situe dans notre contexte derrière les caméras. De ce fait, si on calcule la projection, on va avoir des coordonnées sur l'image fausses.

Pour corriger cela, prenez le point $M_{imu}$ avec $x_{imu} = 3$, $y_{imu} = 0$ et $z_{imu} = 0$, et calculez ses coordonnées dans le repère de la caméra couleur droite

In [ ]:
# A COMPLETER
# Calcul de M_CAM_03 avec les nouvelles coordonnées de M_IMU
...

Votre point ici devrait être dans la caméra ($z_{cam\_03}>0$). On va pouvoir maintenant le projeter dans le plan image de la caméra droite couleur, en suivant la formule suivante:

\begin{equation*}
    \begin{pmatrix} u_{cam\_03} \\ v_{cam\_03} \\ 1 \end{pmatrix} = K_{03} \cdot \begin{pmatrix} x_{cam\_03} \\ y_{cam\_03} \\ z_{cam\_03} \\ 1 \end{pmatrix}
\end{equation*}

Calculez $m_{cam\_03}$, la projection de $M_{cam\_03}$ dans le repère image, et affichez le résultat.

In [ ]:
# A COMPLETER
# Calcul de m_CAM_03
...

Après projection, votre point est exprimé dans le plan image, et en coordonnées homogènes. D'ailleurs, votre point est plus représenté ici sous la forme suivante:

\begin{equation*}
    p_{cam\_03} = \begin{pmatrix} w \cdot u_{cam\_03} \\ w \cdot v_{cam\_03} \\ w \end{pmatrix}
\end{equation*}

Ainsi, pour retrouver les coordonnées de la projection du point dans l'image, il suffit de diviser les coordonnées trouvées par w. A vous de jouer !

In [ ]:
# A COMPLETER
# Calcul des coordonnées de la projection
...

**_QUESTION :_** Est-ce que la projection du point se trouve dans l'image ?

*Note: vous pouvez répondre à cette question via une info dans le fichier de calibration calib_cam_to_cam.txt (chargée avec l'objet Calibration)* 

**_REPONSE :_**

---

## 4) Projection LiDAR

La théorie, c'est cool, mais la pratique, c'est plus sympa. L'objectif ici sera de projeter les points LiDAR obtenus sur une image d'une des caméras. Pour commencer, chargez une image du dossier *2011_09_26_drive_0005_extract*, et corrigez sa distorsion avec les données de calibration

*Note: si votre image est en couleur, pensez bien à la convertir en RGB !*

In [ ]:
# A COMPLETER
# Chargement d'une image *extract* et correction de sa distorsion
...

On travaille ici sur un dossier non synchronisé. A partir des timestamp du LiDAR, et celui de l'image que vous avez choisie, retrouvez le nuage de point le plus proche temporellement de votre image, et chargez le fichier avec la fonction suivante

In [ ]:
def load_velodyne_txt(filepath):
    return np.loadtxt(filepath, dtype=np.float32)

# A COMPLETER
# Chargement du nuage de point LiDAR le plus proche temporellement de l'image choisie
scan = ...

On va analyser d'abord les valeurs de chaque colonne du nuage de point.

In [ ]:
print(f"Scan LiDAR :\n{scan}")
print(f"\nTaille du scan : {scan.shape}")
print(f"\nValeurs 1ère colonne : \n{np.unique(scan[:,0])}")
print(f"\nValeurs 2ère colonne : \n{np.unique(scan[:,1])}")
print(f"\nValeurs 3ère colonne : \n{np.unique(scan[:,2])}")
print(f"\nValeurs 4ère colonne : \n{np.unique(scan[:,3])}")

Les 3 premières colonnes correspondent aux coordonnées cartésiennes du point, dans le repère LiDAR. La 4ème colonne indique ici pour chaque point la réflectance du point, à savoir la quantité lumineuse reçue pour ce point par rapport à celle émise (entre 0 et 100%). 

Afin d'avoir une meilleure visualisation du nuage de point, utilisez la fonction *export_point_cloud_to_html* du fichier *utils.py* pour générer un fichier HTML traçant votre nuage de point.

*Note : exportez plusieurs visualisations du même nuage de point mais en changeant l'information de colorisation*

In [ ]:
from utils import export_point_cloud_to_html

# A COMPLETER
# Export du nuage de point dans un fichier HTML
...

**_QUESTION :_** Remarquez-vous des points aberrants ?

**_REPONSE :_**

Vous avez votre image, et votre nuage de point. On va donc pouvoir commencer la projection !

Calculez dans un premier temps la projection du nuage de point dans le repère le caméra gauche grayscale (00)

*Note: votre nuage de point doit d'abord être modifié pour supprimer la réflectance, et convertir les coordonnées cartésiennes en homogènes*

In [ ]:
# A COMPLETER
# Projection du nuage de point dans le repère de la caméra gauche grayscale 00
...

Passez maintenant ce nuage de point dans le repère de votre caméra

In [ ]:
# A COMPLETER
# Projection du nuage de point dans le repère de la caméra utilisée
...

Filtrez maintenant les points pour ne sélectionnez que ceux devant la caméra ($z>0$). 

Pour cela, il va falloir que vous utilisez un masque, à savoir un vecteur/matrice de booléen construit selon une condition. Dans notre cas, vous avez à suivre ces deux étapes:

- Construction du masque, un vecteur de taille N, où chaque valeur indique si la profondeur z d'un point est positive
- Indexation/séléction du scan avec le masque

In [ ]:
# A COMPLETER
# Filtrage des points LiDAR pour ne garder que ceux devant la caméra
...

**_QUESTION :_** Combien de points reste-t-il après filtrage ? Est-ce que c'est cohérent ?

**_REPONSE :_**

Projetez maintenant le nuage de point filtré dans le plan image de la caméra, et convertissez les coordonnées homogènes en cartésiennes

In [ ]:
# A COMPLETER
# Projection du nuage de point filtré dans le plan image de la caméra, et conversion homogènes => cartésiennes
...

Dernière étape : il va falloir filtrer une nouvelle fois les points, mais pour ne garder que ceux dans les limites de l'image. Via les informations de calibration, et l'utilisation d'un masque, filtrez le nuage de point projeté

In [ ]:
# A COMPLETER
# Filtrage du nuage de points pour ne garder que les points placés dans l'image
...

Bravo, vous avez votre nuage de points projeté ! Bon, pour être sûr que tout va bien, il va falloir le tracer sur l'image...

*C'est long et pas très intéressant pour avoir un rendu sympa, donc voici la fonction*

In [ ]:
def display_lidar_projection(
    image,
    uv,
    depth=None,
    point_size=2.0,
    cmap= "viridis",
    ax=None,
):
    """
    Affiche des points projetés sur une image déjà chargée.

    Parameters
    ----------
    image : np.ndarray
        Image déjà chargée, typiquement de forme (H, W, 3).
    uv : np.ndarray
        Coordonnées image de forme (N, 2).
    depth : np.ndarray | None
        Valeur utilisée pour colorer les points. Si None, couleur uniforme.
    point_size : float
        Taille des points affichés.
    cmap : str
        Colormap Matplotlib utilisée si depth est fourni.
    ax : matplotlib.axes.Axes | None
        Axe existant. Si None, une nouvelle figure est créée.
    """
    created_ax = False
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 5))
        created_ax = True

    ax.imshow(image)

    if depth is None:
        ax.scatter(uv[:, 0], uv[:, 1], s=point_size)
    else:
        scatter = ax.scatter(uv[:, 0], uv[:, 1], c=depth, s=point_size, cmap=cmap)
        plt.colorbar(scatter, ax=ax, label="Depth (m)")

    ax.set_xlim(0, image.shape[1])
    ax.set_ylim(image.shape[0], 0)
    ax.set_title("Projection LiDAR sur image")
    ax.set_xlabel("u [px]")
    ax.set_ylabel("v [px]")

    if created_ax:
        plt.show()

In [ ]:
# A COMPLETER
# Tracé du nuage de point projeté sur l'image
...

Bon, difficile à dire si le nuage de point est bien projeté ici, puisqu'il nous manque une information importante: la profondeur ! 

La fonction proposée ci-dessus a une option *depth* pour coloriser le nuage de point selon la profondeur du point. Seulement ici, il nous manque cette information... Enfin, on l'avait, mais on l'a oublié !

Ici, on va utiliser comme profondeur la coordonnée *z* du nuage de point dans le repère de la caméra utilisée. Récupérez cette coordonnée pour chaque point, et avec les précédents masques créés, faites une nouvelle visualisation du nuage de point avec la profondeur colorisée.

In [ ]:
# A COMPLETER
# Visualisation du nuage de point projeté colorisé en fonction de la profondeur du point
...

**_QUESTION :_** Est-ce que la projection est correcte ? Sinon, qu'est-ce qui peut expliquer le résultat obtenu ?

**_REPONSE :_**

---

## 5) Projection avec caméras rectifiées (Optionnel)


Bon, on a vu comment faire une projection classique. Mais maintenant, comment ça se passe avec les caméras rectifiées ?

En effet, on avait évoqué très rapidement les variables $S_{rect}$, $P_{rect}$ et $R_{rect}$. Alors, à quoi servent-elles ?

Dans notre contexte, le système de caméras mis en place permet de faire de la stéréovision, à savoir estimer la profondeur des objets via 2 points de vues différents (exactement comme ce que vous faites constamment avec vos 2 yeux). Pour que la stéréovision puisse se faire, une correction des images est à apporter pour aligner les lignes épipolaires. Je ne vais pas rentrer dans le détail, mais ce qu'il faut comprendre, c'est que cette correction modifie indéniablement les matrices de calibration. 

Après correction, on obtient donc 3 matrices de calibration:

- $S_{rect}$: la taille de l'image après correction
- $R_{rect}$: la matrice de rotation qui permet de passer du repère de la caméra d'origine vers un repère rectifié, dans lequel toutes les caméras sont alignées (axes parallèles). Cette étape ne modifie pas la position de la caméra, mais uniquement son orientation
- $P_{rect}$: la matrice de projection finale, qui permet de projeter un point 3D exprimé dans le repère caméra rectifié vers le plan image. Elle regroupe les paramètres intrinsèques de la caméra rectifiée (focale, centre optique) ainsi que la position relative de la caméra (notamment la baseline pour la stéréo)

Vous notez qu'on ne trouve plus de coefficients de distorsion. En effet, la rectification des images inclut également la correction de la distorsion.

Concrètement, les opérations réalisées sont proches de celles de notre exercice précédent. En supposant que vos points sont exprimés dans le repère de la caméra d'origine (caméra 00 non rectifiée), la projection dans le plan image de la caméra XX non rectifiée se faisait précédemment comme ceci:

\begin{equation*}
    \begin{pmatrix} u_{cam\_XX} \\ v_{cam\_XX} \\ 1 \end{pmatrix} = K_{cam\_XX} \cdot \begin{pmatrix} R_{XX} & T_{XX} \\ 0_3 & 1 \end{pmatrix} \cdot \begin{pmatrix} x_{cam\_00} \\ y_{cam\_00} \\ z_{cam\_00} \\ 1 \end{pmatrix}
\end{equation*}

Dans le cas des images rectifiées, l'opération de projection dans le plan image de la caméra XX rectifiée se fait de la manière suivante:

\begin{equation*}
    \begin{pmatrix} u_{cam\_XX} \\ v_{cam\_XX} \\ 1 \end{pmatrix} = P_{cam\_XX} \cdot \begin{pmatrix} R_{XX} & 0_3^T \\ 0_3 & 1 \end{pmatrix} \cdot \begin{pmatrix} x_{cam\_00} \\ y_{cam\_00} \\ z_{cam\_00} \\ 1 \end{pmatrix}
\end{equation*}

Les nuages de points LiDAR ont également été traités comme expliqué précédemment, et sont stockés dans un format *.bin*. Je vous passe une fonction pour charger ces fichiers correctement:

In [ ]:
def load_velodyne_bin(filepath):
    return np.fromfile(filepath, dtype=np.float32).reshape(-1, 4)

Hop, vous avez toutes les infos en main. Réalisez maintenant une projection d'un nuage de point sur une image provenant d'un échantillon synchronisé et rectifié (dossier *sync*)

In [ ]:
# A COMPLETER
# Projection d'un nuage de point sur une image dans le cas rectifié/synchronisé
...

---

## 6) Cartographie LiDAR

On en a fini avec la caméra aujourd'hui... Mais avec le LiDAR, on peut encore faire mieux ! Avec le dossier synchronisé, on a à disposition pour chaque timestamp:
- Un nuage de point LiDAR 3D 360° bien propre toi-même tu sais
- La position GPS corrigée
- L'orientation IMU sur 3 axes

Avec toutes ces informations, nous sommes en capacité de créer une carte globale de l'environnement sur toute la séquence !! *Il n'y a que moi qui suis excité ?*

Bref, pour ce dernier exercice, on va y aller pas à pas, comme d'habitude, pour une fusion LiDAR/GPS/IMU. Il faut avoir de l'ambition dans la vie :D

Pour créer notre carte, il va nous falloir un repère global, appelé **repère monde**. Ce repère est invariant dans le temps, et donc ne dépend pas du véhicule. En effet, lorsque le véhicule roule, les repères du LiDAR et de l'OXTS (GPS/IMU) bougent indéniablement. De ce fait, afin de pouvoir situer correctement les points entre plusieurs scans, où le véhicule est différement placé, il est donc nécessaire d'avoir ce repère monde.

Maintenant, comment construit-on ce repère monde ? Et c'est là que le GPS et l'IMU entrent en jeu. Grâce aux données de position (GPS) et d'orientation (IMU), on est capable de placer l'OXTS, et donc le véhicule, dans le repère monde. 

Les opérations de projection sont donc les suivantes, pour placer un point $M$ du repère LiDAR au repère monde:
\begin{equation*}
    M_{velo} \begin{array}{c} \longrightarrow  \\ velo\_to\_imu \end{array} M_{imu} \begin{array}{c} \longrightarrow  \\ imu\_to\_world \end{array} M_{world}
\end{equation*}

Commençons d'abord par charger les données, à savoir le premier scan LiDAR et les données OXTS correspondantes (enfin, une partie des données).

Chargez les données LiDAR, et les données OXTS avec les fonctions *load_velodyne_bin* (partie 5) et *load_oxts* fournies, et affichez les infos OXTS

In [ ]:
# A COMPLETER
# Chargement du scan
...

In [ ]:
def load_oxts(filepath):
    data = np.loadtxt(filepath)

    return {
        "lat": data[0],
        "lon": data[1],
        "alt": data[2],
        "roll": data[3],
        "pitch": data[4],
        "yaw": data[5],
    }

# A COMPLETER
# Chargement des infos OXTS (IMU/GPS)
...

**_QUESTION :_** Quelles informations OXTS ont été chargées ? Dans quelles unités ?

*Note : Vous avez dans le dossier oxts/ un fichier dataformat.txt*

**_REPONSE :_**

La première étape consiste à projeter les points LiDAR dans le repère IMU. On a déjà construit auparavant la matrice de projection *RT_imu_to_velo*, pour faire la projection mais dans l'autre sens. Du coup, il nous suffit de calculer la matrice de projection *RT_velo_to_imu*, la matrice inverse de *RT_imu_to_velo*. 

Grâce à la fonction *np.linalg.inv* pour inverser une matrice, projetez les points LiDAR dans le repère IMU.

*Note : N'oubliez pas de convertir les coordonnées cartésiennes en homogènes, et supprimer la colonne réflectance*

In [ ]:
# A COMPLETER
# Projection des points LiDAR dans le repère IMU
...

La prochaine étape consiste en la projection du repère IMU au repère monde. Mais pour ça, il vous faut la matrice de projection IMU -> Monde...

Commençons par la rotation. Vous avez les orientations de l'IMU, à savoir:
- *roll*, la rotation autour de l'axe x, en degrés
- *pitch*, la rotation autour de l'axe y, en degrés
- *yaw*, la rotation autour de l'axe z, en degrés

Ces orientations nous sont bien utiles: pour passer du repère IMU au repère monde, il suffit donc d'effectuer 3 rotations:
- Une rotation autour de l'axe x d'angle *roll*
- Une rotation autour de l'axe y d'angle *pitch*
- Une rotation autour de l'axe z d'angle *yaw*

Construisez la matrice de rotation repère IMU => repère monde via ces informations

In [ ]:
# A COMPLETER
# Construction de la matrice de rotation repère IMU => repère monde
...

La rotation, c'est fait. Maintenant, passons à la translation. Et pour cela, on va utiliser les coordonnées GPS. Cependant, ces dernières doivent être transformées:
- Les latitudes et longitudes sont des angles, et non des distances en mètres
- L'altitude est une distance en mètre en revanche, mais souvent très petite (en dizaine de mètres)

On pourrait alors faire une conversion polaire 3D en cartésien 3D, comme déjà vu ce semestre. Cependant, cette conversion dans notre contexte perdra de la précision. En effet, une altitude de 20m, et une distance entre un point LiDAR et le véhicule d'environ 50m sont négligeables par rapport au rayon de la Terre (6378 km). 

A la place, on va utiliser une projection plane locale de Mercator, en considérant que la surface où on évolue est plate.

*Non, on est pas des platistes, c'est juste une approximation*

Dès lors, les axes x et z sont calculés en fonction de la latitude et de la longitude, mais également une échelle *scale*, un facteur calculé en fonction de la latitude de la première pose du véhicule. L'altitude ici est utilisée comme coordonnée de l'axe z.

Bref, vous avez à disposition la fonction *gps_to_cartesian* pour automatiquement faire la conversion des coordonnées GPS en coordonnées cartésiennes.

In [ ]:
def gps_to_cartesian(latitude, longitude, altitude, scale):
    """
    Conversion des coordonnées GPS en coordonnées cartésiennes en utilisant la projection plane de Mercator
    Arguments:
    - latitude: latitude du point (en degrés)
    - longitude: longitude du point (en degrés)
    - altitude: altitude du point (en mètres)
    - scale: facteur calculé en fonction de la latitude de la pose initiale (cos(lat_0))
    Sortie: Coordonnées cartésiennes (x,y,z) (en mètres)
    """
    
    rayon_terre = 6378137.0 # Rayon de la Terre en mètres (au niveau de l'équateur)

    x = scale * longitude * np.pi * rayon_terre / 180.0
    y = scale * rayon_terre * np.log(np.tan((90.0 + latitude) * np.pi / 360.0))
    z = altitude

    return np.array([x, y, z])

Calculez les coordonnées cartésiennes du repère IMU via les coordonnées GPS, avec $scale = cos(lat_0)$, où $lat_0$ est la latitude de la première pose

*Attention: np.cos prend en entrée des angles en degrés ou radians ?*

In [ ]:
# A COMPLETER
# Construction de l'échelle scale
...

# Construction du vecteur de translation à partir des coordonnées GPS converties en cartésien
...

On a notre matrice de rotation et notre vecteur de translation, on peut donc construire notre matrice de projection de la manière suivante :

\begin{equation*}
    RT_{imu\_to\_world} = \begin{pmatrix} R_{imu\_to\_world} & T_{imu\_to\_world} \\ 0_3 & 1 \end{pmatrix}
\end{equation*}

Avec $R_{imu\_to\_world}$ la matrice de rotation créée à partir des orientations IMU, et $T_{imu\_to\_world}$ le vecteur de translation créé à partir des coordonnées GPS converties en cartésien.

In [ ]:
# A COMPLETER
# Construction de la matrice de projection IMU => monde
...

Effectuez maintenant la projection des points situés dans le repère IMU dans le repère monde, et affichez les valeurs obtenues dans chaque colonne (via la fonction *np.unique*)

In [ ]:
# A COMPLETER
# Projection du repère IMU au repère monde, et affichage des valeurs obtenues dans chaque colonne
...

**_QUESTION :_** Que remarquez-vous ?

**_REPONSE :_**

Vous remarquez donc que le repère monde est certes invariant, mais très excentré de la scène principale (plusieurs milliers de km): le centre de votre repère se situe... au centre de la Terre !

Alors, en quoi c'est dérangeant ? Le souci principal réside dans la précision : si vous avez des valeurs très élevées, ça nécessite de les coder avec un type de données pouvant les prendre en charge, offrant potentiellement moins de précision. Il est donc préférable de recentrer cette scène. Et pour cela, on va introduire un nouveau repère, appelé **repère local**

*"Fais chier, encore un repère..."*

Alors, le repère local, c'est tout simplement le repère de la 1ère pose du véhicule (donc pour le premier scan LiDAR). L'idée ici est de considérer que le véhicule commence au centre de notre repère, et donc tous les scans suivants seront exprimés en fonction de la toute première position.

Le processus de projection pour exprimer un point $M$ du repère LiDAR au repère local devient donc le suivant:

\begin{equation*}
    M_{velo} \begin{array}{c} \longrightarrow  \\ velo\_to\_imu \end{array} M_{imu} \begin{array}{c} \longrightarrow  \\ imu\_to\_world \end{array} M_{world} \begin{array}{c} \longrightarrow  \\ world\_to\_local \end{array} M_{local}
\end{equation*}

Pour passer du repère monde au repère local, il suffit donc de calculer la matrice de projection qui est tout simplement l'inverse de la matrice de projection IMU => monde pour la 1ère pose.

Calculez la matrice de projection du repère monde au repère local, et par conséquent la matrice de projection du repère IMU au repère local.

**_QUESTION :_** Pour la première pose, que devrait-on obtenir comme matrice de projection IMU => local ?

In [ ]:
# A COMPLETER
# Construction de la matrice de projection repère monde => repère local 
...

# Construction de la matrice de projection repère IMU => repère local 
...

**_REPONSE :_**

La projection du repère monde au repère local résout le soucis de décentrage, mais pas forcément de la précision. Lorsqu'on projette dans le repère monde, les distances obtenues font perdre de la précision, comme expliqué précédemment. De ce fait, il est préférable d'utiliser directement la projection du repère IMU au repère local, en calculant comme précédemment la matrice de projection. De ce fait le processus de projection final (promis, on ne modifie plus ensuite), est le suivant:

\begin{equation*}
    M_{velo} \begin{array}{c} \longrightarrow  \\ velo\_to\_imu \end{array} M_{imu} \begin{array}{c} \longrightarrow  \\ imu\_to\_local \end{array} M_{local}
\end{equation*}


Projetez les points LiDAR du repère IMU au repère local

In [ ]:
# A COMPLETER
# Projection des points du repère IMU au repère Local
...

La dernière étape consiste donc à faire ces projections sur tous les scans LiDAR, et de concaténer ces scans en un seul gros nuage de points.

Je vous donne la structure de code, à vous de compléter avec ce que vous avez fait précédemment pour construire le nuage de points global. Il y'a du post-traitement afin d'extraire la trajectoire également (pour l'afficher sur le nuage de points), et surtout réduire drastiquement le nombre de points afin d'avoir un nuage de points lisible

In [ ]:
# A COMPLETER
# Construction du nuage de point global

import os
from utils import filter_lidar_points, voxel_downsample, export_global_cloud_with_trajectory_html

path_lidar = ... # Le chemin où se situe toutes les acquisitions LiDAR
path_oxts = ... # Le chemin où se situe toutes les acquisitions OXTS
ids_samples = sorted([f.replace(".bin","") for f in os.listdir(path_lidar)])


# Calcul des infos communes pour tous les scans en fonction de la 1ère pose

# Chargement des infos OXTS de la 1ère pose
first_pose = load_oxts(path_oxts+ids_samples[0]+'.txt')

# Calcul de l'échelle scale pour la conversion GPS => Cartésien
scale = ... 

# Calcul de la matrice de projection du repère monde au repère local, à partir de la première pose
RT_world_to_local = ...

# Calcul de la matrice de projection du repère LiDAR au repère IMU
RT_velo_to_imu = ...

all_points = []
trajectory = []

for id in ids_samples:
    scan = load_velodyne_bin(path_lidar+id+'.bin')
    oxts = load_oxts(path_oxts+id+'.txt')
    
    # Calcul de la matrice de projection du repère IMU au repère monde
    RT_imu_to_world = ...
    
    # Calcul de la matrice de projection du repère IMU au repère local
    RT_imu_to_local = ...
    
    # Conversion des points LiDAR de coordonnées cartésiennes en coordonnées homogènes
    scan_xyz_h = ...

    # Projection des points du repère LiDAR au repère local
    scan_xyz_local_h = ...
    
    # Concaténation des points dans le nuage de point global
    all_points.append(scan_xyz_local_h[:,:3])
    trajectory.append(RT_imu_to_local[:3, 3])
    
global_points = np.vstack(all_points)
trajectory = np.array(trajectory)

global_points = voxel_downsample(global_points, voxel_size=0.5)
print(f"Taille de global_points: {global_points.shape}")

html_path = export_global_cloud_with_trajectory_html(global_points, trajectory, "point_cloud_global.html", color_by="z")
print(f"Fichier généré : {html_path}")